In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import math

from torch.utils.data import random_split
import torch.optim as optim

In [ ]:

# ==========================================
# Preprocessing
# ==========================================
print("Loading dataset...")
df = pd.read_csv('data/final_data_tranmissions_v2.csv')

# Dropping columns
cols_to_drop = ['dstId', 'srcSat', 'dstSat', 'loraTP', 'loraCF', 'loraSF', 'loraBW', 'loraCR', 'satId', 'srcId']
df = df.drop(columns=cols_to_drop)

df.head()

Loading dataset...


,pid,time,latDev,longDev,elevSat,doppler,rcvOk,alt,raan,id_simulation
0,745,3.07362,59.7066,2.2184,6.97644,19609.9,1,600,190,sim_alt_600_raan_190
1,767,10.24260,60.6759,-8.8517,5.31067,17899.3,0,600,190,sim_alt_600_raan_190
2,769,10.40190,55.5044,26.9474,4.64867,16660.0,0,600,190,sim_alt_600_raan_190
3,782,17.27580,53.9628,25.9511,2.84052,16987.4,1,600,190,sim_alt_600_raan_190
4,794,22.74790,58.2060,26.7352,8.19765,16628.3,0,600,190,sim_alt_600_raan_190


In [ ]:
# Sort by id_simulaiton and time
df = df.sort_values(by=['id_simulation', 'time']).reset_index(drop=True)

# Selecting continuous features
continuous_features = ['latDev', 'longDev', 'elevSat', 'doppler', 'alt', 'raan']

# Apply StandardScaler 
scaler = StandardScaler()
df[continuous_features] = scaler.fit_transform(df[continuous_features])
df.head()

,pid,time,latDev,longDev,elevSat,doppler,rcvOk,alt,raan,id_simulation
0,745,3.07362,1.231036,-0.716948,-0.560581,1.280646,1,-1.039041,-1.262421,sim_alt_600_raan_170
1,767,10.24260,1.351017,-1.575096,-0.560273,1.413901,1,-1.039041,-1.262421,sim_alt_600_raan_170
2,769,10.40190,0.710881,1.200031,-1.037195,0.537491,0,-1.039041,-1.262421,sim_alt_600_raan_170
3,782,17.27580,0.520060,1.122798,-1.139264,0.595954,1,-1.039041,-1.262421,sim_alt_600_raan_170
4,794,22.74790,1.045289,1.183581,-0.820518,0.496215,0,-1.039041,-1.262421,sim_alt_600_raan_170


# Sequence Generation

In [ ]:
SEQ_LENGTH = 16
seq_X   = []
label_y = []

for sim_id, group_df in df.groupby('id_simulation'):
    print('sim_id: ', sim_id)
    print('group_df: ', group_df.shape)

    num_features_array  = group_df[continuous_features].values
    ###cat_features_array  = group_df['srcId'].values
    time_array          = group_df['time'].values
    target_array        = group_df['rcvOk'].values
    num_packets         = len(group_df)
    
    # Discard if the simulation has less than SEQ_LENGHTS
    if num_packets < SEQ_LENGTH:
        continue
    
    print('-' * 50)
    print('num_packets: ' , num_packets)    

    count_seq = 0
    # Create sliding windows - Ventanas deslizantes
    for i in range(num_packets - SEQ_LENGTH + 1):
        
        # Seleccionar la ventana de la iteración
        # Select the window        
        window_num      = num_features_array[i : i + SEQ_LENGTH]
        ###window_cat      = cat_features_array[i : i + SEQ_LENGTH].reshape(-1, 1)
        window_times    = time_array[i : i + SEQ_LENGTH]
        window_targets  = target_array[i : i + SEQ_LENGTH]
        
        # El paquete central a predecir es el último de la ventana
        # The prediction should be the last elemento of the window
        target_idx      = SEQ_LENGTH - 1
        label_target    = window_targets[target_idx]
        time_target     = window_times[target_idx]
        
        # --- CALCULAR EL DELTA DE TIEMPO (CLAVE PARA EL ENCODING) ---
        # Si un paquete ocurrió en el mismo segundo que el objetivo, delta_t = 0        
        delta_t = (window_times - time_target).reshape(-1, 1)
        #delta_t =  time_target - window_times
        #print('delta_t: ', delta_t)

        # CONCATENAR TODO EN UNA SOLA MATRIZ: [Numéricas (6), Categórica (1), Tiempo (1)]
        # Total features por paquete = 8
        #window_X = np.hstack((window_num, window_cat, delta_t))
        window_X = np.hstack((window_num, delta_t))

        seq_X.append(window_X)
        label_y.append(label_target)

        count_seq = count_seq + 1
    
    print('count_seq: ' , count_seq)

sim_id:  sim_alt_600_raan_170
group_df:  (222, 10)
--------------------------------------------------
num_packets:  222
count_seq:  207
sim_id:  sim_alt_600_raan_180
group_df:  (268, 10)
--------------------------------------------------
num_packets:  268
count_seq:  253
sim_id:  sim_alt_600_raan_190
group_df:  (283, 10)
--------------------------------------------------
num_packets:  283
count_seq:  268
sim_id:  sim_alt_600_raan_200
group_df:  (280, 10)
--------------------------------------------------
num_packets:  280
count_seq:  265
sim_id:  sim_alt_600_raan_210
group_df:  (266, 10)
--------------------------------------------------
num_packets:  266
count_seq:  251
sim_id:  sim_alt_850_raan_300
group_df:  (29, 10)
--------------------------------------------------
num_packets:  29
count_seq:  14
sim_id:  sim_alt_850_raan_310
group_df:  (61, 10)
--------------------------------------------------
num_packets:  61
count_seq:  46
sim_id:  sim_alt_850_raan_320
group_df:  (117, 10)
---

In [ ]:
# Convertir a Tensores
# Usamos float32 para todo. La variable categórica la pasaremos a entero (long) dentro del modelo
X_tensor = torch.tensor(np.array(seq_X), dtype=torch.float32)
y_tensor = torch.tensor(np.array(label_y), dtype=torch.float32)

print(f"Tensor X unificado: {X_tensor.shape}") # Resultado esperado: (2503, 16, 8)
print(f"Tensor y: {y_tensor.shape}")

# DataLoader más limpio
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

Tensor X unificado: torch.Size([2503, 16, 7])
Tensor y: torch.Size([2503])


# Positional Encoding

In [ ]:
class TimePositionalEncoding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        # Frecuencias fijas según Attention Is All You Need
        # Fixed frequencies according to Attention is all you need        
        # div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        i = torch.arange(0, d_model//2)
        div_term = torch.exp(-np.log(10000) * (2*i / d_model))
        self.register_buffer('div_term', div_term)

    def forward(self, delta_t):
        # batch x seq x d_model
        pe = torch.zeros(delta_t.size(0), delta_t.size(1), self.d_model, device=delta_t.device)
        # angle = (batch x seq x 1) x ( 1 x 1 x d_model//2)
        angle = delta_t.unsqueeze(-1) * self.div_term.unsqueeze(0).unsqueeze(0)
        pe[:, :, 0::2] = torch.sin(angle)
        pe[:, :, 1::2] = torch.cos(angle)
        return pe

# Transformer

In [ ]:
class LoraCollisionTransformer(nn.Module):        
    def __init__(self, num_numerical_features, d_model=64, n_heads=4, n_layers=2, dropout=0.1):
        # num_numerical_features- cantidad de variables fisicas (features)    
        # d_model               - Es la dimensión de representación interna del Transformer. Piensa en esto como la cantidad de "canales de información" que el modelo 
        #                         usará internamente para describir cada paquete. Todo el procesamiento profundo se hará en esta dimensión.
        # n_heads               - Le dice al Transformer en cuántas perspectivas diferentes debe dividir su atención.
        # n_layers              - Cuántas veces se repite el proceso de análisis (cuántos bloques Transformer se apilan).
        # dropout               - 
        super().__init__()
        self.num_num_features = num_numerical_features
        self.d_model = d_model
        
        # 1. Proyección directa de las variables físicas a d_model
        # Entran 'num_numerical_features' (ej. 6) y salen 'd_model' (ej. 64)
        # Recibe tu vector original de 6 variables físicas y lo transforma (lo proyecta) matemáticamente en un vector de tamaño d_model (64 dimensiones).
        # Recives the original input vector and project it to a vector size d_model
        self.input_projection = nn.Linear(num_numerical_features, d_model)
        
        # Apply TimePositionalEncoding 
        self.time_encoding = TimePositionalEncoding(d_model)
        
        # Apply encoder_layer
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, 
                                                   dim_feedforward=d_model*4, dropout=dropout, batch_first=True)        
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x shape: (Batch, Seq_Length, Features = 7) -> 6 físicas + 1 Delta_t
        
        # 1. DESEMPAQUETAR EL TENSOR X
        # Tomamos todas las columnas excepto la última (que es el tiempo)
        x_num = x[:, :, :self.num_num_features] 
        
        # La última columna es el Delta de Tiempo
        delta_t = x[:, :, -1] 
        
        # 2. PROYECCIÓN FÍSICA
        # Convertimos las 6 variables físicas en un vector de 64 dimensiones
        x_projected = self.input_projection(x_num)       
        
        # 3. POSITIONAL ENCODING (Inyección del tiempo)
        time_pe = self.time_encoding(delta_t)
        x_encoded = x_projected + time_pe
        
        # 4. TRANSFORMER
        transformer_out = self.transformer_encoder(x_encoded) 
        
        # 5. CLASIFICACIÓN
        target_packet = transformer_out[:, -1, :]             
        output = self.classifier(target_packet)               
        
        return output.squeeze(-1)

# Preparing Training

In [ ]:

# 80% training and 20% testing
train_size = int(0.8 * len(dataset))
test_size  = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create DataLoaders
BATCH_SIZE   = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training data: {len(train_dataset)} sequences")
print(f"Testing data: {len(test_dataset)} sequences")

Training data: 2002 sequences
Testing data: 501 sequences


In [ ]:
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "cpu"
print(f"Training ing : {device}")

# Create the model 
model = LoraCollisionTransformer(
    num_numerical_features=6, 
    d_model=64, 
    n_heads=4, # number of projections
    n_layers=2, # 1 number of repetitions 
    dropout=0.1
).to(device)

# Loss for binary classification 
# Función de pérdida para clasificación Binaria (0 o 1)
# Binary Cross Entropy
criterion = nn.BCELoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

Training ing : cpu


In [ ]:
EPOCHS = 20

for epoch in range(EPOCHS):
    
    # ==========================
    # Training
    # ==========================
    model.train() # Activate training mode
    train_loss      = 0.0
    correct_train   = 0
    total_train     = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Clean gradient 
        optimizer.zero_grad()
        
        # Forward pass
        predictions = model(batch_X) # Usa internamente el squeeze(-1)
        
        # Calculate error loss
        loss = criterion(predictions, batch_y)
        
        # Backward pass: Calcular cómo debe ajustarse cada neurona
        # Backpropagation
        loss.backward()
        
        # Optimización: Actualizar los pesos matemáticos
        # Update weights
        optimizer.step()
                
        # loss.item - The average of the batch losses will give you an estimate of the “epoch loss” during training. Returns the value of this tensor as a standard Python number
        # 
        train_loss += loss.item() * batch_X.size(0)
        
        # Convert to 0 - 1 the vector batch
        predictions_class = (predictions >= 0.5).float()
        
        correct_train += (predictions_class == batch_y).sum().item()
        total_train   += batch_y.size(0)
        
    avg_train_loss = train_loss / total_train
    train_acc = correct_train / total_train
    
    # ==========================
    # Testing
    # ==========================
    model.eval()
    test_loss    = 0.0
    correct_test = 0
    total_test   = 0
        
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            # Predict
            predictions = model(batch_X)
            loss        = criterion(predictions, batch_y)
            test_loss   += loss.item() * batch_X.size(0)
            
            # Calculate correct or not 
            predictions_class = (predictions >= 0.5).float()
            correct_test      += (predictions_class == batch_y).sum().item()
            total_test        += batch_y.size(0)
            
    avg_test_loss = test_loss / total_test
    test_acc      = correct_test / total_test
    
    # Imprimir resumen del Epoch
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} - Acc: {train_acc*100:.1f}% | "
          f"Test Loss: {avg_test_loss:.4f} - Acc: {test_acc*100:.1f}%")

Epoch 01/20 | Train Loss: 0.1917 - Acc: 91.7% | Test Loss: 0.4051 - Acc: 85.4%
Epoch 02/20 | Train Loss: 0.1923 - Acc: 91.7% | Test Loss: 0.3961 - Acc: 84.6%
Epoch 03/20 | Train Loss: 0.1711 - Acc: 93.1% | Test Loss: 0.3952 - Acc: 82.8%
Epoch 04/20 | Train Loss: 0.1859 - Acc: 92.9% | Test Loss: 0.3781 - Acc: 84.4%
Epoch 05/20 | Train Loss: 0.1883 - Acc: 92.7% | Test Loss: 0.3381 - Acc: 85.4%
Epoch 06/20 | Train Loss: 0.1683 - Acc: 93.7% | Test Loss: 0.3578 - Acc: 85.0%
Epoch 07/20 | Train Loss: 0.1598 - Acc: 94.1% | Test Loss: 0.3760 - Acc: 85.0%
Epoch 08/20 | Train Loss: 0.1485 - Acc: 94.2% | Test Loss: 0.4582 - Acc: 84.8%
Epoch 09/20 | Train Loss: 0.1719 - Acc: 93.4% | Test Loss: 0.3592 - Acc: 86.4%
Epoch 10/20 | Train Loss: 0.1296 - Acc: 94.7% | Test Loss: 0.4109 - Acc: 84.6%
Epoch 11/20 | Train Loss: 0.1528 - Acc: 94.3% | Test Loss: 0.3794 - Acc: 85.2%
Epoch 12/20 | Train Loss: 0.1233 - Acc: 95.0% | Test Loss: 0.3821 - Acc: 84.6%
Epoch 13/20 | Train Loss: 0.1298 - Acc: 95.0% | Test

# Fine-Tuning

In [ ]:
import math
import itertools
import time

In [ ]:
''' 
# Create the model 
model = LoraCollisionTransformer(
    num_numerical_features=6, 
    d_model=64, 
    n_heads=4, # number of projections
    n_layers=2, # 1 number of repetitions 
    dropout=0.1
).to(device)

Epoch 01/20 | Train Loss: 0.6142 - Acc: 66.7% | Test Loss: 0.5463 - Acc: 73.9%
Epoch 02/20 | Train Loss: 0.5308 - Acc: 74.8% | Test Loss: 0.4966 - Acc: 77.4%
Epoch 03/20 | Train Loss: 0.4810 - Acc: 78.6% | Test Loss: 0.4877 - Acc: 76.8%
Epoch 04/20 | Train Loss: 0.4645 - Acc: 78.1% | Test Loss: 0.4546 - Acc: 79.6%
Epoch 05/20 | Train Loss: 0.4302 - Acc: 79.9% | Test Loss: 0.4656 - Acc: 77.0%
Epoch 06/20 | Train Loss: 0.4252 - Acc: 78.5% | Test Loss: 0.4248 - Acc: 80.0%
Epoch 07/20 | Train Loss: 0.3889 - Acc: 80.9% | Test Loss: 0.4042 - Acc: 82.6%
Epoch 08/20 | Train Loss: 0.3817 - Acc: 81.8% | Test Loss: 0.3831 - Acc: 82.6%
Epoch 09/20 | Train Loss: 0.3719 - Acc: 82.2% | Test Loss: 0.4151 - Acc: 81.8%
Epoch 10/20 | Train Loss: 0.3692 - Acc: 82.1% | Test Loss: 0.4133 - Acc: 82.4%
Epoch 11/20 | Train Loss: 0.3446 - Acc: 84.5% | Test Loss: 0.4058 - Acc: 80.6%
Epoch 12/20 | Train Loss: 0.3154 - Acc: 85.2% | Test Loss: 0.4571 - Acc: 81.6%
Epoch 13/20 | Train Loss: 0.3252 - Acc: 85.2% | Test Loss: 0.4196 - Acc: 80.8%
Epoch 14/20 | Train Loss: 0.3081 - Acc: 85.7% | Test Loss: 0.4124 - Acc: 81.0%
Epoch 15/20 | Train Loss: 0.2921 - Acc: 87.4% | Test Loss: 0.3997 - Acc: 83.8%
Epoch 16/20 | Train Loss: 0.2680 - Acc: 87.8% | Test Loss: 0.4167 - Acc: 81.8%
Epoch 17/20 | Train Loss: 0.2760 - Acc: 87.9% | Test Loss: 0.3762 - Acc: 84.6%
Epoch 18/20 | Train Loss: 0.2611 - Acc: 88.3% | Test Loss: 0.3939 - Acc: 84.0%
Epoch 19/20 | Train Loss: 0.2601 - Acc: 88.3% | Test Loss: 0.3594 - Acc: 84.0%
Epoch 20/20 | Train Loss: 0.2410 - Acc: 89.1% | Test Loss: 0.3719 - Acc: 83.2%

'''

In [ ]:
''' 
# Create the model 
model = LoraCollisionTransformer(
    num_numerical_features=6, 
    d_model=64, 
    n_heads=4, # number of projections
    n_layers=1, # 1 number of repetitions 
    dropout=0.1
).to(device)


Epoch 01/20 | Train Loss: 0.6186 - Acc: 66.3% | Test Loss: 0.5560 - Acc: 73.5%
Epoch 02/20 | Train Loss: 0.5437 - Acc: 74.5% | Test Loss: 0.5308 - Acc: 75.2%
Epoch 03/20 | Train Loss: 0.5154 - Acc: 76.0% | Test Loss: 0.5186 - Acc: 77.4%
Epoch 04/20 | Train Loss: 0.5158 - Acc: 76.3% | Test Loss: 0.5552 - Acc: 72.1%
Epoch 05/20 | Train Loss: 0.4790 - Acc: 78.1% | Test Loss: 0.4599 - Acc: 78.4%
Epoch 06/20 | Train Loss: 0.4583 - Acc: 78.7% | Test Loss: 0.4565 - Acc: 79.0%
Epoch 07/20 | Train Loss: 0.4427 - Acc: 80.5% | Test Loss: 0.4514 - Acc: 78.2%
Epoch 08/20 | Train Loss: 0.4297 - Acc: 79.8% | Test Loss: 0.4883 - Acc: 74.9%
Epoch 09/20 | Train Loss: 0.4270 - Acc: 80.4% | Test Loss: 0.4400 - Acc: 79.0%
Epoch 10/20 | Train Loss: 0.4184 - Acc: 80.6% | Test Loss: 0.4398 - Acc: 78.8%
Epoch 11/20 | Train Loss: 0.3942 - Acc: 81.1% | Test Loss: 0.4312 - Acc: 79.0%
Epoch 12/20 | Train Loss: 0.3845 - Acc: 81.3% | Test Loss: 0.4554 - Acc: 79.4%
Epoch 13/20 | Train Loss: 0.3960 - Acc: 82.0% | Test Loss: 0.4357 - Acc: 79.2%
Epoch 14/20 | Train Loss: 0.3709 - Acc: 81.5% | Test Loss: 0.4170 - Acc: 79.2%
Epoch 15/20 | Train Loss: 0.3724 - Acc: 82.8% | Test Loss: 0.4014 - Acc: 80.8%
Epoch 16/20 | Train Loss: 0.3742 - Acc: 82.5% | Test Loss: 0.4149 - Acc: 81.4%
Epoch 17/20 | Train Loss: 0.3514 - Acc: 83.8% | Test Loss: 0.4519 - Acc: 77.4%
Epoch 18/20 | Train Loss: 0.3585 - Acc: 83.5% | Test Loss: 0.4707 - Acc: 75.0%
Epoch 19/20 | Train Loss: 0.3560 - Acc: 83.5% | Test Loss: 0.4348 - Acc: 80.0%
Epoch 20/20 | Train Loss: 0.3480 - Acc: 83.3% | Test Loss: 0.3905 - Acc: 80.4%

'''

In [2]:
#def load_and_prepare_data():
seq_length=16
print("Loading dataset...")
csv_path = 'data/final_data_tranmissions_v2.csv'
df       = pd.read_csv(csv_path)

# Dropping columns
cols_to_drop = ['dstId', 'srcSat', 'dstSat', 'loraTP', 'loraCF', 'loraSF', 'loraBW', 'loraCR', 'satId', 'srcId']
df = df.drop(columns=cols_to_drop)

# Sort by id_simulaiton and time
df = df.sort_values(by=['id_simulation', 'time']).reset_index(drop=True)

# Selecting continuous features
continuous_features = ['latDev', 'longDev', 'elevSat', 'doppler', 'alt', 'raan']

# Apply StandardScaler 
scaler = StandardScaler()
df[continuous_features] = scaler.fit_transform(df[continuous_features])

Loading dataset...


In [3]:
df.head()

,pid,time,latDev,longDev,elevSat,doppler,rcvOk,alt,raan,id_simulation
0,745,3.07362,1.231036,-0.716948,-0.560581,1.280646,1,-1.039041,-1.262421,sim_alt_600_raan_170
1,767,10.24260,1.351017,-1.575096,-0.560273,1.413901,1,-1.039041,-1.262421,sim_alt_600_raan_170
2,769,10.40190,0.710881,1.200031,-1.037195,0.537491,0,-1.039041,-1.262421,sim_alt_600_raan_170
3,782,17.27580,0.520060,1.122798,-1.139264,0.595954,1,-1.039041,-1.262421,sim_alt_600_raan_170
4,794,22.74790,1.045289,1.183581,-0.820518,0.496215,0,-1.039041,-1.262421,sim_alt_600_raan_170


In [ ]:



    
seq_X   = []
label_y = []


for sim_id, group_df in df.groupby('id_simulation'):
    num_features_array  = group_df[continuous_features].values
    ###cat_features_array  = group_df['srcId'].values
    time_array          = group_df['time'].values
    target_array        = group_df['rcvOk'].values
    num_packets         = len(group_df)
    
    # Discard if the simulation has less than SEQ_LENGHTS
    if num_packets < seq_length:
        continue
                
    # Create sliding windows - Ventanas deslizantes
    for i in range(num_packets - seq_length + 1):
        print('*'*50)
        # Seleccionar la ventana de la iteración
        # Select the window        
        window_num      = num_features_array[i : i + seq_length]
        print('window_num: ', window_num.shape)
        ###window_cat      = cat_features_array[i : i + SEQ_LENGTH].reshape(-1, 1)
        window_times    = time_array[i : i + seq_length]
        print('window_times: ', window_times.shape)
        window_targets  = target_array[i : i + seq_length]
        print('window_targets: ', window_targets.shape)
        
        # El paquete central a predecir es el último de la ventana
        # The prediction should be the last elemento of the window
        target_idx      = seq_length - 1
        label_target    = window_targets[target_idx]
        time_target     = window_times[target_idx]
                    
        # Si un paquete ocurrió en el mismo segundo que el objetivo, delta_t = 0        
        delta_t = (window_times - time_target).reshape(-1, 1)
        #delta_t =  time_target - window_times
        #print('delta_t: ', delta_t)
        
        #window_X = np.hstack((window_num, window_cat, delta_t))
        window_X = np.hstack((window_num, delta_t))

        seq_X.append(window_X)
        label_y.append(label_target)
            
X = torch.tensor(np.array(seq_X), dtype=torch.float32)
y = torch.tensor(np.array(label_y), dtype=torch.float32)


window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  (16, 6)
window_num:  